In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the states
states = ['Bull Market', 'Bear Market', 'Neutral Market']

# Define the actions
actions = ['Buy', 'Sell', 'Hold']

# Generate dynamic transition probabilities (e.g., based on market volatility)
def generate_transition_probs(volatility=0.2):
    return {
        'Buy': {
            'Bull Market': [0.7 - volatility, 0.1 + volatility, 0.2],
            'Bear Market': [0.2 + volatility, 0.6 - volatility, 0.2],
            'Neutral Market': [0.3, 0.3, 0.4]
        },
        'Sell': {
            'Bull Market': [0.6 - volatility, 0.1, 0.3 + volatility],
            'Bear Market': [0.3, 0.5, 0.2 + volatility],
            'Neutral Market': [0.4, 0.3, 0.3]
        },
        'Hold': {
            'Bull Market': [0.8, 0.1 - volatility, 0.1 + volatility],
            'Bear Market': [0.1, 0.7, 0.2 + volatility],
            'Neutral Market': [0.3, 0.2, 0.5]
        }
    }

# Generate rewards dynamically based on market condition (higher reward in bull market)
def generate_rewards():
    return {
        'Bull Market': {'Buy': 10, 'Sell': 5, 'Hold': 2},
        'Bear Market': {'Buy': -10, 'Sell': 1, 'Hold': -1},
        'Neutral Market': {'Buy': 0, 'Sell': 0, 'Hold': 1}
    }

# Dynamic transition probabilities
transition_probs = generate_transition_probs()

# Rewards
rewards = generate_rewards()

# Discount factor
gamma = 0.9

# Initialize value function
value_function = {state: 0 for state in states}

# Tracking convergence history
value_iteration_history = []
policy_iteration_history = []

# Define the value iteration function
def value_iteration(threshold=0.01, max_iterations=1000):
    iteration = 0
    while iteration < max_iterations:
        delta = 0
        new_value_function = value_function.copy()

        # For each state
        for state in states:
            state_values = []
            # For each action
            for action in actions:
                expected_value = 0
                # Compute the expected value for taking this action
                for next_state, prob in zip(states, transition_probs[action][state]):
                    reward = rewards[state][action]
                    expected_value += prob * (reward + gamma * value_function[next_state])
                state_values.append(expected_value)

            # Update the value function
            new_value_function[state] = max(state_values)
            delta = max(delta, abs(new_value_function[state] - value_function[state]))

        value_function.update(new_value_function)
        value_iteration_history.append(new_value_function.copy())  # Track history
        iteration += 1

        # Check for convergence
        if delta < threshold:
            break

    return value_function, iteration

# Define the policy extraction function
def extract_policy(value_function):
    policy = {}
    for state in states:
        state_action_values = []
        for action in actions:
            expected_value = 0
            for next_state, prob in zip(states, transition_probs[action][state]):
                reward = rewards[state][action]
                expected_value += prob * (reward + gamma * value_function[next_state])
            state_action_values.append(expected_value)

        best_action = actions[np.argmax(state_action_values)]
        policy[state] = best_action

    return policy

# Perform value iteration
optimal_value_function, value_iteration_iterations = value_iteration()

# Extract the optimal policy from the value function
optimal_policy_vi = extract_policy(optimal_value_function)

# Define the policy iteration function
def policy_iteration(max_iterations=1000):
    # Initialize with a random policy
    policy = {state: np.random.choice(actions) for state in states}
    stable = False
    iteration = 0

    while iteration < max_iterations and not stable:
        # Policy evaluation
        policy_value_function = value_function.copy()
        for _ in range(100):  # Fixed number of iterations for evaluation
            new_value_function = policy_value_function.copy()
            for state in states:
                action = policy[state]
                expected_value = 0
                for next_state, prob in zip(states, transition_probs[action][state]):
                    reward = rewards[state][action]
                    expected_value += prob * (reward + gamma * policy_value_function[next_state])
                new_value_function[state] = expected_value
            policy_value_function = new_value_function.copy()

        # Policy improvement
        new_policy = {}
        for state in states:
            state_action_values = []
            for action in actions:
                expected_value = 0
                for next_state, prob in zip(states, transition_probs[action][state]):
                    reward = rewards[state][action]
                    expected_value += prob * (reward + gamma * policy_value_function[next_state])
                state_action_values.append(expected_value)

            best_action = actions[np.argmax(state_action_values)]
            new_policy[state] = best_action

        # Check if the policy is stable
        if policy == new_policy:
            stable = True
        policy = new_policy.copy()
        policy_iteration_history.append(policy_value_function.copy())  # Track history
        iteration += 1

    return policy, iteration

# Perform policy iteration
optimal_policy_pi, policy_iteration_iterations = policy_iteration()

# Visualization function to compare convergence
def plot_convergence():
    vi_convergence = [sum(vf.values()) for vf in value_iteration_history]
    pi_convergence = [sum(vf.values()) for vf in policy_iteration_history]

    plt.plot(vi_convergence, label='Value Iteration Convergence')
    plt.plot(pi_convergence, label='Policy Iteration Convergence')
    plt.title('Convergence of Value Iteration vs Policy Iteration')
    plt.xlabel('Iterations')
    plt.ylabel('Sum of Value Function')
    plt.legend()
    plt.grid(True)
    plt.show()

# Output the results
print("Value Iteration Results:")
print(f"Optimal Value Function: {optimal_value_function}")
print(f"Optimal Policy (Value Iteration): {optimal_policy_vi}")
print(f"Value Iteration converged in {value_iteration_iterations} iterations.\n")

print("Policy Iteration Results:")
print(f"Optimal Policy (Policy Iteration): {optimal_policy_pi}")
print(f"Policy Iteration converged in {policy_iteration_iterations} iterations.\n")

# Comparison
print("Comparison of Convergence:")
print(f"Value Iteration: {value_iteration_iterations} iterations.")
print(f"Policy Iteration: {policy_iteration_iterations} iterations.")



Value Iteration Results:
Optimal Value Function: {'Bull Market': 109.6147758292124, 'Bear Market': 120.53045602240107, 'Neutral Market': 98.62576484020143}
Optimal Policy (Value Iteration): {'Bull Market': 'Buy', 'Bear Market': 'Sell', 'Neutral Market': 'Sell'}
Value Iteration converged in 161 iterations.

Policy Iteration Results:
Optimal Policy (Policy Iteration): {'Bull Market': 'Buy', 'Bear Market': 'Sell', 'Neutral Market': 'Sell'}
Policy Iteration converged in 1 iterations.

Comparison of Convergence:
Value Iteration: 161 iterations.
Policy Iteration: 1 iterations.
